# 🎙️ AIOS Google Colab Free Whisper Audio Transcriber (Whisper Large-v3 GPU)

Этот ноутбук запускает **100% БЕСПЛАТНЫЙ высокоскоростной сервер расшифровки аудио/звонков** на базе модели **Whisper Large-v3** на T4 GPU Google Colab.

### 🚀 Инструкция по запуску:
1. Убедитесь, что вы включили GPU: **`Среда выполнения` ➔ `Сменить тип среды выполнения` ➔ `T4 GPU`**.
2. Запустите **Ячейку 1** для установки зависомостей (`faster-whisper`, `fastapi`, `cloudflared`).
3. Запустите **Ячейку 2** для загрузки модели `large-v3` на GPU и старта API-сервера.
4. Запустите **Ячейку 3** для создания бесплатного Cloudflare-туннеля и получения URL сервиса.

In [ ]:
# === ЯЧЕЙКА 1: Установка зависомостей ===
!pip install faster-whisper fastapi uvicorn python-multipart cloudflared pydantic -q
print('✅ Все зависимости Whisper успешно установлены!')

In [ ]:
# === ЯЧЕЙКА 2: Создание и запуск Whisper FastAPI сервера на GPU ===
import os
import sys
import tempfile
import threading
import uvicorn
from fastapi import FastAPI, File, UploadFile, Query, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from faster_whisper import WhisperModel

app = FastAPI(title='AIOS Colab Whisper Transcriber', version='1.0.0')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

print('🚀 Загрузка модели Whisper Large-v3 на T4 GPU (FP16)...')
# Модель large-v3 загружается за ~15 секунд и занимает всего 3 ГБ VRAM из 16 ГБ на бесплатном Colab GPU
whisper_model = WhisperModel('large-v3', device='cuda', compute_type='float16')
print('✅ Модель Whisper Large-v3 успешно загружена на GPU!')

@app.get('/health')
def health():
    return {
        'status': 'ok',
        'service': 'aios-colab-whisper',
        'model': 'large-v3',
        'device': 'cuda',
        'free_tier': True
    }

@app.post('/transcribe')
async def transcribe_audio(
    file: UploadFile = File(...),
    language: str = Query(None, description='Код языка (ru, uk, en и т.д.) или None для автоопределения'),
    task: str = Query('transcribe', description='transcribe или translate'),
    beam_size: int = Query(5, description='Размер луч поиска')
):
    try:
        suffix = os.path.splitext(file.filename)[1] or '.mp3'
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
            content = await file.read()
            tmp.write(content)
            tmp_path = tmp.name

        segments_gen, info = whisper_model.transcribe(
            tmp_path,
            language=language if language and language != 'auto' else None,
            task=task,
            beam_size=beam_size,
            vad_filter=True
        )

        segments = []
        full_text_list = []
        for seg in segments_gen:
            full_text_list.append(seg.text.strip())
            segments.append({
                'id': seg.id,
                'start': round(seg.start, 2),
                'end': round(seg.end, 2),
                'text': seg.text.strip(),
                'avg_logprob': round(seg.avg_logprob, 3)
            })

        os.remove(tmp_path)
        full_text = ' '.join(full_text_list)

        return {
            'status': 'success',
            'filename': file.filename,
            'language': info.language,
            'language_probability': round(info.language_probability, 3),
            'duration_seconds': round(info.duration, 2),
            'transcription': full_text,
            'segments_count': len(segments),
            'segments': segments
        }
    except Exception as err:
        raise HTTPException(status_code=500, detail=str(err))

def start_server():
    uvicorn.run(app, host='0.0.0.0', port=8000)

t = threading.Thread(target=start_server, daemon=True)
t.start()
print('⚡ FastAPI Whisper Server запущен на порту 8000!')

In [ ]:
# === ЯЧЕЙКА 3: Создание публичного туннеля для AIOS ===
import subprocess, re, time

print('📡 Запуск туннеля Cloudflare...')
tunnel = subprocess.Popen('cloudflared tunnel --url http://localhost:8000', shell=True, stderr=subprocess.PIPE, text=True)

time.sleep(5)
for line in iter(tunnel.stderr.readline, ''):
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print('\n🎉 ========================================================')
            print('🔗 ПУБЛИЧНЫЙ URL ВАШЕГО WHISPER СЕРВЕРА В GOOGLE COLAB:')
            print(f'   {tunnel_url}')
            print('========================================================\n')
            print('👉 Выполните эту команду на сервере AIOS для регистрации:')
            print(f'python3 scripts/register_colab_whisper.py {tunnel_url}')
            print('========================================================\n')
            break